# ReviewerMatch — Colab (ingest → GPU embed → FAISS)

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. Run the **setup** cell below and edit `OPENALEX_EMAIL`, `REPO_URL`, `TARGET_AUTHOR_COUNT`.
3. Run remaining cells in order. Full ~150k ingestion may take **1–3+ hours**; start with **5_000** to validate.
4. Download **`reviewermatch_bundle.tar.gz`**, extract into your repo’s **`data/`** folder, then locally run **`python -m scripts.load_metadata`**.

Vector labels in FAISS are **`1 … N`**, equal to **`faiss_row_id`** in `authors_meta.json` and **`Author.id`** in the DB after load.


In [ ]:
# If the embedding step says CPU-only, uncomment and rerun this cell once:
# !pip install -q torch --index-url https://download.pytorch.org/whl/cu124
!pip install -q sentence-transformers faiss-cpu httpx tenacity tqdm numpy

import os

# --- EDIT THESE ---
os.environ["OPENALEX_EMAIL"] = "you@example.com"
TARGET_AUTHOR_COUNT = 150_000   # e.g. 5000 for a first dry run
REPO_URL = "https://github.com/YOUR_USER/reviewermatch-api.git"
# ------------------

assert "@" in os.environ["OPENALEX_EMAIL"], "Set a real email for OpenAlex polite pool"
print("TARGET_AUTHOR_COUNT =", TARGET_AUTHOR_COUNT)


In [ ]:
import subprocess
from pathlib import Path

ROOT = Path("/content/reviewermatch-api")
if not ROOT.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True, cwd="/content")

%cd /content/reviewermatch-api

assert (ROOT / "data" / "ingest_openalex.py").is_file(), "Expected data/ingest_openalex.py — fix REPO_URL"
Path("data").mkdir(parents=True, exist_ok=True)
print("Repo:", ROOT.resolve())


In [ ]:
import asyncio
import os
from pathlib import Path

# Ingestion (resumable: re-run continues appending to data/authors_raw.jsonl)
from data.ingest_openalex import main as ingest_main

out = Path("data/authors_raw.jsonl")
asyncio.run(
    ingest_main(
        os.environ["OPENALEX_EMAIL"],
        out,
        target_count=int(TARGET_AUTHOR_COUNT),
    )
)
print("Wrote:", out.resolve(), "size_mb=", round(out.stat().st_size / (1024**2), 2))


In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → GPU"

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

authors = []
texts = []
with open("data/authors_raw.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        a = json.loads(line)
        authors.append(a)
        texts.append((a.get("profile_text") or "")[:2000])

print(f"Embedding {len(texts)} profiles…")

embeddings = model.encode(
    texts,
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print("shape:", embeddings.shape, "dtype:", embeddings.dtype)
assert embeddings.shape[1] == 384, "all-MiniLM-L6-v2 must be 384-dim"


In [ ]:
import faiss
import numpy as np

n, dim = embeddings.shape
# Labels1..n match Author.id after scripts/load_metadata.py
row_ids = np.arange(1, n + 1, dtype=np.int64)

base = faiss.IndexFlatIP(dim)
index = faiss.IndexIDMap(base)
index.add_with_ids(embeddings, row_ids)

Path("data").mkdir(parents=True, exist_ok=True)
faiss.write_index(index, "data/authors.faiss")
print("FAISS vectors:", index.ntotal, "dim:", dim)


In [ ]:
import json
from pathlib import Path

metadata = []
for i, a in enumerate(authors):
    rid = i + 1  # must match FAISS id and Author.id
    works = a.get("recent_works") or []
    years = [w.get("year") for w in works if w.get("year")]
    last_y = max(years) if years else None
    metadata.append({
        "faiss_row_id": rid,
        "openalex_id": a["openalex_id"],
        "display_name": a["display_name"],
        "orcid": a.get("orcid"),
        "works_count": a.get("works_count", 0),
        "cited_by_count": a.get("cited_by_count", 0),
        "h_index": a.get("h_index", 0),
        "i10_index": a.get("i10_index", 0),
        "two_year_mean_citedness": a.get("two_year_mean_citedness", 0.0),
        "institution_name": a.get("institution_name"),
        "institution_country": a.get("institution_country"),
        "top_concepts": a.get("top_concepts") or [],
        "recent_works": works,
        "profile_text": a.get("profile_text"),
        "last_known_activity_year": last_y,
    })

out = Path("data/authors_meta.json")
with out.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False)
print("records:", len(metadata), "→", out.resolve())
assert len(metadata) == len(authors) == index.ntotal


In [ ]:
import os
import tarfile
from pathlib import Path

bundle = Path("/content/reviewermatch_bundle.tar.gz")
with tarfile.open(bundle, "w:gz") as tar:
    tar.add("data/authors.faiss", arcname="authors.faiss")
    tar.add("data/authors_meta.json", arcname="authors_meta.json")

print("Bundle:", bundle.resolve(), "MB:", round(bundle.stat().st_size / (1024**2), 2))

try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print("Not on Colab — copy", bundle, "manually")


## After download (on your laptop)

```bash
cd reviewermatch-api
tar xzvf ~/Downloads/reviewermatch_bundle.tar.gz -C data/
ls -lh data/authors.faiss data/authors_meta.json
cp .env.example .env   # set DATABASE_URL if needed
python -m scripts.load_metadata
uvicorn app.main:app --reload
```

Expect **`data/authors.faiss`** (~230MB per100k vectors, scales ~linearly) and **`data/authors_meta.json`** (tens of MB with `profile_text`).
